# WangchanBERTa + LST20 NER — 2026 Modernization

> **Project history:** the original Thai NER experiments in this repository were created in **2022**. This notebook is an explicit **2026 continuation / modernization update**, using a cleaner data pipeline, current Hugging Face training APIs, automated metrics, and CI-tested utilities.

The notebook intentionally does **not** redistribute LST20. Download the corpus from its authorized source, extract it locally, then point `DATA_DIR` to the extracted `LST20Corpus` folder.

## 1. Install the 2026 project environment

Run this notebook from a cloned copy of the repository. The package includes the current Transformers v5-style training API; a full LST20 fine-tuning run is intended for a GPU or a longer CPU run.

In [ ]:
%pip install -q -r requirements-2026.txt

## 2. Configure the local LST20 path

In [ ]:
from pathlib import Path

DATA_DIR = Path("/path/to/LST20Corpus")
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

print("LST20 path:", DATA_DIR)
print("Model:", MODEL_NAME)

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Set DATA_DIR to an extracted authorized LST20Corpus directory before running: {DATA_DIR}"
    )

## 3. Load LST20 with the 2026 parser

In [ ]:
from thai_ner_2026.data import load_lst20_dataset_dict

# Use small limits first to validate your environment.
dataset = load_lst20_dataset_dict(
    DATA_DIR,
    train_limit=100,
    validation_limit=100,
    test_limit=100,
)
dataset


## 4. Load WangchanBERTa and align NER labels

The modern pipeline uses a fast tokenizer so source-word labels can be mapped reliably to subword tokens.

In [ ]:
from transformers import AutoTokenizer
from thai_ner_2026.labels import LABEL2ID
from thai_ner_2026.preprocessing import tokenize_and_align_batch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
assert tokenizer.is_fast, "A fast tokenizer is required for label alignment."

tokenized = dataset.map(
    lambda batch: tokenize_and_align_batch(batch, tokenizer, LABEL2ID, max_length=256),
    batched=True,
    remove_columns=dataset["train"].column_names,
)
tokenized


## 5. Train

For a complete training run, the repository exposes a CLI so the experiment is reproducible outside the notebook:

In [ ]:
# Example command — remove the leading # after setting DATA_DIR.
# !thai-ner-train --data-dir /path/to/LST20Corpus --epochs 3 --output-dir outputs/wangchanberta-lst20-2026


## 6. Evaluation contract

The 2026 pipeline reports entity-level **precision, recall, F1**, plus token accuracy. LST20 end tags (`E_*`) are normalized to IOB2 only for `seqeval` scoring; the original 31-class labels remain the training targets.

A full benchmark table should only be added after a complete, recorded LST20 training run.